# Graph Attention, Pooling & Over-smoothing

Companion notebook for the [Graph Attention lesson](https://ml-viz-ruby.vercel.app/courses/graph-neural-networks/03-graph-attention-and-pooling).

We implement **graph attention** (softmax over a node's neighbors), a permutation-invariant
**graph-level readout**, and demonstrate **over-smoothing** by measuring how node embeddings
converge as we stack layers. Pure NumPy.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

A = np.zeros((5, 5))
for u, v in [(0,1),(1,2),(2,3),(3,4),(1,3)]:
    A[u,v] = A[v,u] = 1
A = A + np.eye(5)                 # self-loops
X = rng.normal(size=(5, 4))

## Intuition — let nodes decide which neighbors matter

GCN weights every neighbor by degree alone; **graph attention (GAT)** lets each node *learn* how much
to listen to each neighbor — the same softmax attention as transformers, restricted to graph edges.
For graph-level tasks (classify a whole molecule), a permutation-invariant **readout** (sum/mean/max
over nodes) pools node embeddings into one vector. And the depth ceiling from lesson 1 —
**over-smoothing** — is measured here directly, with the pairwise-distance collapse. We verify the
attention's key identity and show the standard mitigation.

## 1 — Attention over neighbors

For node v we score each neighbor u, softmax the scores **within v's neighborhood**, and take the
weighted sum. The weights α sum to 1 per node — the model learns which neighbors matter.

In [ ]:
def leaky_relu(x, a=0.2):
    return np.where(x > 0, x, a * x)

def graph_attention(A, H, W, a_vec):
    n = A.shape[0]
    HW = H @ W                                    # transform features
    out = np.zeros_like(HW)
    alphas = np.zeros((n, n))
    for v in range(n):
        nb = np.where(A[v] > 0)[0]
        scores = np.array([leaky_relu(a_vec @ np.concatenate([HW[v], HW[u]])) for u in nb])
        e = np.exp(scores - scores.max())
        w = e / e.sum()                           # softmax over THIS node's neighbors
        alphas[v, nb] = w
        out[v] = (w[:, None] * HW[nb]).sum(0)
    return out, alphas

d = 4
W = rng.normal(size=(d, d)) * 0.5
a_vec = rng.normal(size=2 * d) * 0.5
_, alphas = graph_attention(A, X, W, a_vec)
print('attention weights (row v = how much v attends to each neighbor):\n', alphas)
print('each row sums to 1:', alphas.sum(1))

**What to notice:** each node computes a score for every neighbor (a tiny MLP on the concatenated
pair), softmaxes **over its own neighborhood** (rows sum to 1), and takes the weighted sum. Unlike
GCN's fixed `1/√(dᵤdᵥ)` weights, these `α`s depend on the *features* — the graph version of attention's
content-based routing.

## The library way — attention generalizes the mean

The sanity identity: with the attention vector zeroed, every score ties and the softmax becomes
**uniform** — GAT must reduce exactly to mean aggregation of the transformed features. Verify it.

In [ ]:
out0, alphas0 = graph_attention(A, X, W, np.zeros(2 * d))
HW = X @ W
manual_mean = np.stack([HW[np.where(A[v] > 0)[0]].mean(0) for v in range(A.shape[0])])
assert np.allclose(out0, manual_mean), "zero attention vector must reduce GAT to the neighbor mean"
print('with a = 0, every neighbor gets equal weight:', alphas0[1][alphas0[1] > 0].round(3))
print('GAT(a=0) == mean aggregation ✓  (attention strictly generalizes the GCN-style mean)')

**What to notice:** zeroing the attention parameters recovers plain mean aggregation exactly — GAT
is a strict generalization: it *can* express GCN's uniform weighting, and training moves it away only
where uneven weighting helps. This nesting is also a great unit test for any GAT implementation.

## 2 — Graph-level readout

To predict a property of the *whole* graph, pool all node embeddings with a permutation-invariant
function. We verify the readout is unchanged when nodes are relabeled.

In [ ]:
def readout(H, kind='mean'):
    return {'sum': H.sum(0), 'mean': H.mean(0), 'max': H.max(0)}[kind]

perm = np.array([3, 0, 4, 1, 2])
for kind in ['sum', 'mean', 'max']:
    assert np.allclose(readout(X, kind), readout(X[perm], kind))
print('\u2713 sum/mean/max readouts are permutation-invariant (graph-level vector is order-free)')
print('graph embedding (mean readout):', readout(X, 'mean'))

**What to notice:** the sum/mean/max **readout** collapses any number of nodes, in any order, into
one fixed-size vector — verified permutation-invariant. This is the bridge from node embeddings to
whole-graph predictions (molecule properties, program classification).

## 3 — Over-smoothing: deeper collapses embeddings

We repeatedly apply mean aggregation and measure the average pairwise distance between node
embeddings. It shrinks toward zero — past a few layers all nodes look the same.

In [ ]:
def mean_aggregate(A, H):
    return (A @ H) / A.sum(1, keepdims=True)

def avg_pairwise_dist(H):
    n = len(H)
    return np.mean([np.linalg.norm(H[i] - H[j]) for i in range(n) for j in range(i+1, n)])

H = X.copy()
print(f'layer 0: avg pairwise distance = {avg_pairwise_dist(H):.4f}')
for k in range(1, 11):
    H = mean_aggregate(A, H)
    print(f'layer {k:2d}: avg pairwise distance = {avg_pairwise_dist(H):.4f}')
print('\nDistances collapse -> over-smoothing. This caps practical GNNs at 2-4 layers.')

**What to notice:** the average pairwise distance between node embeddings **shrinks monotonically**
with depth — by layer 10 the nodes are nearly indistinguishable. This is over-smoothing measured, and
it's why GNN depth stalls where CNN/transformer depth doesn't.

## Gotchas & tradeoffs

- **Attention costs per-edge compute** and (multi-head) parameters — worth it on heterogeneous
  neighborhoods, wasted on uniform ones.
- **Over-smoothing is the depth ceiling** — residual connections (`H + aggregate(H)`) keep node
  identity alive (demo below); still, 2–4 layers is typical.
- **Readout choice matters:** mean for size-invariant properties, sum when counts matter (GIN), max
  for detecting presence of a motif.
- **Attention weights ≠ explanations:** like transformer attention, `α`s are suggestive but not
  faithful attributions.

In [ ]:
# Residual connections slow over-smoothing: H <- 0.5*H + 0.5*aggregate(H)
H_plain, H_res = X.copy(), X.copy()
print(f'{"layer":>6} {"plain":>10} {"residual":>10}')
for k in range(1, 11):
    H_plain = mean_aggregate(A, H_plain)
    H_res = 0.5 * H_res + 0.5 * mean_aggregate(A, H_res)
    if k in (1, 3, 5, 10):
        print(f'{k:>6} {avg_pairwise_dist(H_plain):>10.4f} {avg_pairwise_dist(H_res):>10.4f}')
print('\n-> the residual keeps embeddings separated far longer (same trick as ResNets)')

**What to notice:** with a residual mix, the pairwise distances decay far more slowly — node
identities survive extra layers. It's the same `+x` identity-path from the ResNet lesson, now defending
against homogenization instead of vanishing gradients. The deep-learning toolbox keeps reusing its
best trick.

## ✏️ Your turn

**Exercise.** Implement `attention_weights(scores)` — a numerically-stable softmax over a 1-D array
of neighbor scores — and `is_oversmoothed(H, tol)` returning `True` when the average pairwise
distance between node embeddings has dropped below `tol` (the collapse signature).

In [ ]:
def attention_weights(scores):
    # TODO(you): numerically-stable softmax (subtract the max before exp), returns weights summing to 1
    return ...

def is_oversmoothed(H, tol=0.05):
    # TODO(you): True if avg_pairwise_dist(H) < tol
    return ...

In [ ]:
# This assert cell passes silently when your implementation is correct.
w = attention_weights(np.array([1.0, 2.0, 3.0]))
assert np.isclose(w.sum(), 1.0) and np.argmax(w) == 2
assert np.allclose(attention_weights(np.array([5.0, 5.0])), [0.5, 0.5])
assert not is_oversmoothed(X)                       # raw features are spread out
H = X.copy()
for _ in range(30):
    H = mean_aggregate(A, H)
assert is_oversmoothed(H)                            # after many layers, collapsed
print('\u2713 softmax attention and over-smoothing detector are correct')

<details>
<summary>Solution</summary>

```python
def attention_weights(scores):
    e = np.exp(scores - scores.max())
    return e / e.sum()

def is_oversmoothed(H, tol=0.05):
    return avg_pairwise_dist(H) < tol
```

GAT's attention is just a softmax restricted to each node's neighbors. Over-smoothing is the price
of depth on graphs: repeated averaging is a low-pass filter that eventually erases all the
node-to-node variation you need to classify.

</details>